# Part 1: Rectified Flow Matching Basics

> **Learning objectives:**
> - Understand rectified flow matching: learning straight-line flows from noise to data
> - Implement a velocity predictor and train it with flow matching
> - Sample using Euler integration with uniform and SD3 schedules
> - Implement classifier-free guidance (CFG)

We work with the **two moons** dataset — a simple 2D distribution that lets us visualize everything.

In [ ]:
import torch
import torch as t
from torch import nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
import sys
from pathlib import Path

exercises_dir = Path(".").resolve().parent
if str(exercises_dir) not in sys.path:
    sys.path.insert(0, str(exercises_dir))

from part1_flow_matching_basics import solutions
from part1_flow_matching_basics.tests import (
    test_velocity_mlp, test_train,
    test_sample_uniform, test_sample_sd3, test_sample_cfg,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Provided: Two Moons Dataset and DataLoader

In [ ]:
def make_two_moons(n_samples=2000, noise=0.04):
    X, y = make_moons(n_samples=n_samples, noise=noise)
    return t.tensor(X, dtype=t.float32), t.tensor(y, dtype=t.long)


def make_dataloader(n_samples=5000, batch_size=256, noise=0.04, label_dropout=0.2):
    """DataLoader with label shift and dropout.
    Labels: 0=unconditional, 1=moon0, 2=moon1. ~20% dropped to 0."""
    data, labels = make_two_moons(n_samples, noise=noise)
    labels = labels + 1  # shift: 0=uncond, 1=moon0, 2=moon1

    class TwoMoonsDataset(t.utils.data.Dataset):
        def __init__(self, data, labels, label_dropout):
            self.data = data
            self.labels = labels
            self.label_dropout = label_dropout

        def __len__(self):
            return len(self.data)

        def __getitem__(self, idx):
            x = self.data[idx]
            c = self.labels[idx].clone()
            if self.label_dropout > 0 and t.rand(1).item() < self.label_dropout:
                c = t.tensor(0, dtype=t.long)
            return x, c

    dataset = TwoMoonsDataset(data, labels, label_dropout)
    return t.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True), data, labels


# Visualize
data, labels = make_two_moons(2000)
plt.figure(figsize=(6, 4))
plt.scatter(data[:, 0], data[:, 1], c=labels, cmap='coolwarm', s=5, alpha=0.7)
plt.title("Two Moons Dataset (labels: 0=blue, 1=red)")
plt.axis('equal')
plt.show()

## Background: Rectified Flow Matching

The idea is beautifully simple. We want to learn a mapping from noise $z \sim \mathcal{N}(0, I)$ to data $x \sim p_{\text{data}}$.

**Rectified flow** defines a straight-line path between noise and data:

$$x_t = x - t \cdot (x - z)$$

- At $t = 0$: $x_0 = x$ (clean data)
- At $t = 1$: $x_1 = z$ (pure noise)

The **velocity** along this path is constant: $v = x - z$.

We train a neural network $v_\theta(x_t, t, c)$ to predict this velocity at any point along the path.

## Exercise 1 — MLP Velocity Predictor
*Difficulty: 🔴🔴⭕⭕⭕ | Importance: 🔵🔵🔵🔵⭕ | ~10 min*

Implement a **velocity predictor** that maps `(x_t, t, c)` to a vector in `R^{data_dim}` with the shapes expected by `test_velocity_mlp`. **Any architecture is fine** as long as the API and tensor shapes match — you are not required to use a gated MLP. The reference solution uses a gated MLP (`output = down(up(inp) * SiLU(gate(inp)))`) for consistency with later parts of the tutorial, but a plain MLP or other blocks are equally acceptable.

`test_velocity_mlp` also checks **sensitivity**: the predicted velocity must actually change when you perturb each coordinate of `x_t`, when you change `t`, and when you switch class labels (so `x_t`, `t`, and `c` must all feed into the network).

The input is `[x_t, t, class_emb(c)]` concatenated. The class embedding maps class indices (0=unconditional, 1=moon0, 2=moon1) to a learned vector.

In [ ]:
class VelocityMLP(nn.Module):
    """Simple MLP that predicts velocity given (x_t, t, class)."""

    def __init__(self, data_dim=2, hidden_dim=128, class_dim=10, n_classes=2):
        super().__init__()
        input_dim = data_dim + 1 + class_dim
        self.class_emb = nn.Embedding(n_classes + 1, class_dim)  # +1 for unconditional (index 0)

        # YOUR CODE HERE
        pass

    def forward(self, x_t, ts, c):
        """
        Args:
            x_t: (batch, data_dim) noisy points
            ts: (batch,) timesteps in [0, 1]
            c: (batch,) class labels (0 = unconditional)
        Returns:
            (batch, data_dim) predicted velocity
        """
        # YOUR CODE HERE
        pass

In [ ]:
test_velocity_mlp(VelocityMLP)

## Exercise 2 — Training Loop
*Difficulty: 🔴🔴🔴⭕⭕ | Importance: 🔵🔵🔵🔵🔵 | ~20 min*

Implement the training loop with flow matching loss inline.

The dataloader yields `(x, c)` batches where labels are already shifted (+1) and 20% are dropped to 0 for CFG.

The model $v_\theta(x_t, t, c)$ takes a noisy point $x_t$, a timestep $t \in [0,1]$, and a class label $c$, and predicts the velocity $v = x - z$.

The **flow matching loss** is the **mean squared error** between the predicted and true velocity:

$$\mathcal{L} = \operatorname{MSE}(v_\theta, v) = \frac{1}{D}\sum_{i=1}^{D} \bigl(v_{\theta,i} - v_i\bigr)^2$$

where $v = x - z$, $x_t = x - t \cdot v$, $z \sim \mathcal{N}(0, I)$, and $D$ is the velocity dimension (here `data_dim`). Implement it as `F.mse_loss(v_pred, v_true)` — that matches the equation and the suggested learning rates. (Some papers instead write $\sum_i (v_{\theta,i} - v_i)^2$ without the $1/D$; that changes gradient scale, so you would need a different learning rate.)

### Logit-normal timestep sampling

Instead of sampling $t \sim U(0,1)$, use **logit-normal sampling**: $t = \text{sigmoid}(\text{randn}())$. This concentrates training on timesteps near $t=0$ and $t=1$ where the velocity field changes rapidly and the model needs the most practice. From the [SD3 paper](https://arxiv.org/abs/2403.03206) — we'll see the same idea in the sampling schedule (Exercise 4).

In [ ]:
def train(model, dataloader, n_steps=5000, lr=1e-3):
    """Train the velocity model.

    Args:
        model: VelocityMLP
        dataloader: yields (points, labels) batches
        n_steps: optimization steps
        lr: learning rate
    Returns:
        list of losses
    """
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    dl_iter = iter(dataloader)

    for step in range(n_steps):
        try:
            x, c = next(dl_iter)
        except StopIteration:
            dl_iter = iter(dataloader)
            x, c = next(dl_iter)

        x = x.to(device)
        c = c.to(device)

        # YOUR CODE HERE
        loss = 0
        losses.append(loss.item())
        pass

    return losses

In [ ]:
test_train(train, VelocityMLP)

In [ ]:
model = VelocityMLP(data_dim=2, hidden_dim=128).to(device)
dataloader, data_train, labels_train = make_dataloader(5000, label_dropout=0.2)
losses = train(model, dataloader, n_steps=5000, lr=1e-2)

plt.figure(figsize=(6, 3))
plt.plot(losses, alpha=0.3)
plt.plot(np.convolve(losses, np.ones(100) / 100, mode='valid'), 'r-')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

## Exercise 3 — Euler Sampling (Uniform Schedule)
*Difficulty: 🔴🔴⭕⭕⭕ | Importance: 🔵🔵🔵🔵🔵 | ~10 min*

Now that we have a trained model, let's generate samples! Integrate the velocity field from $t=1$ (noise) to $t=0$ (data) using Euler's method:

$$z_{i+1} = z_i + (t_i - t_{i+1}) \cdot v_\theta(z_i, t_i, c)$$

Start with a **uniform schedule**.

In [ ]:
@t.no_grad()
def sample_uniform(model, n_samples, data_dim=2, n_steps=50, c=None):
    """Sample using uniform timestep schedule (linspace from 1 to 0)."""
    z = t.randn(n_samples, data_dim, device=next(model.parameters()).device)
    # YOUR CODE HERE
    pass

In [ ]:
test_sample_uniform(sample_uniform, VelocityMLP)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(data_train[:, 0], data_train[:, 1], s=3, alpha=0.5)
axes[0].set_title("Ground Truth")
axes[0].axis('equal')

c = t.ones(2000, dtype=t.long, device=device)  # condition on moon 1
samples = sample_uniform(model, 2000, n_steps=50, c=c).cpu()
axes[1].scatter(samples[:, 0], samples[:, 1], s=3, alpha=0.5)
axes[1].set_title("Uniform Schedule (50 steps)")
axes[1].axis('equal')
plt.tight_layout()
plt.show()

## Exercise 4 — Improved Sampler (SD3 Schedule)
*Difficulty: 🔴🔴⭕⭕⭕ | Importance: 🔵🔵🔵🔵⭕ | ~10 min*

The uniform schedule spaces steps evenly across $[0, 1]$. But is that optimal?

### Why a non-uniform schedule?

Not all parts of the trajectory need equal precision. Near $t=1$ (noise), the model is making critical structural decisions — small errors here compound through the rest of the trajectory. In the middle ($t \approx 0.5$), the trajectory is fairly straight and the velocity is nearly constant, so large steps are fine. Near $t=0$ (data), the model is refining details along a mostly-determined path, so fewer steps suffice.

### The SD3 transformation

$$t'_i = \frac{3 t_i}{2 t_i + 1}$$

The derivative $dt'/dt = 3/(2t+1)^2$ tells us how spacing changes:

- Near $t=1$: derivative is $1/3$, so steps are **3× smaller** (more precision for initial denoising)
- Near $t=0.5$: derivative is $3/4$, so steps are moderately compressed
- Near $t=0$: derivative is $3$, so steps are **3× larger** (less precision needed for final refinement)

Same Euler integration, just a schedule that front-loads precision where errors are most costly.

In [ ]:
@t.no_grad()
def sample_sd3(model, n_samples, data_dim=2, n_steps=50, c=None):
    """Sample using SD3 logit-normal schedule."""
    z = t.randn(n_samples, data_dim, device=next(model.parameters()).device)
    # YOUR CODE HERE
    pass

In [ ]:
test_sample_sd3(sample_sd3, VelocityMLP)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for row, (sampler, name) in enumerate([(sample_uniform, "Uniform"), (sample_sd3, "SD3")]):
    for col, n_steps in enumerate([3, 5, 10, 50]):
        c = t.ones(2000, dtype=t.long, device=device)
        samples = sampler(model, 2000, n_steps=n_steps, c=c).cpu()
        axes[row, col].scatter(samples[:, 0], samples[:, 1], s=3, alpha=0.5)
        axes[row, col].set_title(f"{name}, {n_steps} steps")
        axes[row, col].axis('equal')
        axes[row, col].set_xlim(-2, 3)
        axes[row, col].set_ylim(-1.5, 2)
plt.tight_layout()
plt.show()

## Exercise 5 — Add Classifier-Free Guidance
*Difficulty: 🔴🔴🔴⭕⭕ | Importance: 🔵🔵🔵🔵🔵 | ~15 min*

CFG amplifies conditioning by contrasting conditional and unconditional predictions:

$$v_{\text{guided}} = v_{\text{uncond}} + \gamma \cdot (v_{\text{cond}} - v_{\text{uncond}})$$

- $\gamma = 1$: pure conditional (no guidance)
- $\gamma > 1$: amplified conditioning (sharper, more mode-seeking)
- $\gamma = 0$: pure unconditional

This works because we trained with 20% label dropout — the model learned both conditional and unconditional generation.

In [ ]:
@t.no_grad()
def sample_cfg(model, n_samples, data_dim=2, n_steps=50, c=None, cfg=2.0):
    """Sample with classifier-free guidance using SD3 schedule."""
    z = t.randn(n_samples, data_dim, device=next(model.parameters()).device)
    # YOUR CODE HERE
    pass

In [ ]:
test_sample_cfg(sample_cfg, VelocityMLP)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for row, class_label in enumerate([1, 2]):
    for col, cfg_scale in enumerate([0.0, 1.0, 2.0, 5.0]):
        c = t.full((2000,), class_label, dtype=t.long, device=device)
        samples = sample_cfg(model, 2000, n_steps=50, c=c, cfg=cfg_scale).cpu()
        axes[row, col].scatter(data_train[:, 0], data_train[:, 1],
                               c=(labels_train).numpy(), cmap='coolwarm', s=2, alpha=0.15)
        axes[row, col].scatter(samples[:, 0], samples[:, 1], s=3, alpha=0.5, c='green')
        axes[row, col].set_title(f"Class {class_label}, CFG={cfg_scale}")
        axes[row, col].axis('equal')
        axes[row, col].set_xlim(-2, 3)
        axes[row, col].set_ylim(-1.5, 2)
plt.suptitle("CFG Sweep: green=generated, background=ground truth", y=1.02)
plt.tight_layout()
plt.show()

## Visualize Denoising Trajectories

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

@t.no_grad()
def compute_trajectories(model, n_samples=200, n_steps=100, c=None):
    dev = next(model.parameters()).device
    z = t.randn(n_samples, 2, device=dev)
    ts = t.linspace(1, 0, n_steps + 1, device=dev)
    ts = 3 * ts / (2 * ts + 1)
    trajectory = [z.cpu().clone()]
    for i in range(n_steps):
        t_batch = ts[i] * t.ones(n_samples, device=dev)
        v_pred = model(z, t_batch, c)
        z = z + (ts[i] - ts[i + 1]) * v_pred
        trajectory.append(z.cpu().clone())
    return t.stack(trajectory)


c_traj = t.ones(200, dtype=t.long, device=device)
traj = compute_trajectories(model, n_samples=200, n_steps=100, c=c_traj).numpy()

# Animated denoising video
fig, ax = plt.subplots(figsize=(6, 5))
ax.set_xlim(-3, 4); ax.set_ylim(-2, 3)
ax.set_aspect('equal')
ax.scatter(data_train[:, 0], data_train[:, 1], s=2, alpha=0.15, c='gray', label='Ground truth')
scatter = ax.scatter([], [], s=8, c='blue', alpha=0.6)
title = ax.set_title("")
ax.legend(loc='upper left')

def animate(frame):
    scatter.set_offsets(traj[frame])
    t_val = 1.0 - frame / (len(traj) - 1)
    title.set_text(f"Denoising: step {frame}/{len(traj)-1} (t≈{t_val:.2f})")
    return scatter, title

anim = FuncAnimation(fig, animate, frames=len(traj), interval=50, blit=True)
plt.close()
HTML(anim.to_html5_video())

## Summary

You've learned the fundamentals of rectified flow matching:

- **The flow**: straight-line interpolation $x_t = x - t(x-z)$ between data and noise
- **Training**: predict the velocity $v = x - z$ at random points along the flow, with 20% label dropout for CFG
- **Sampling**: Euler integration from $t=1$ (noise) to $t=0$ (data)
- **SD3 schedule**: $t' = 3t/(2t+1)$ concentrates steps where it matters most
- **CFG**: $v_{\text{guided}} = v_{\text{uncond}} + \gamma(v_{\text{cond}} - v_{\text{uncond}})$ amplifies conditioning

Next: **Part 2** scales this up to images with a Diffusion Transformer (DiT) on MNIST.